## Setup and load data

This is the final consolidated run. I fix the random seeds so the results are reproducible, load the cleaned trajectories, and confirm the GPU. Every number in the thesis comes from this single notebook run.

In [1]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

device="cuda" if torch.cuda.is_available() else "cpu"
print("device:",device)
print("gpu:",torch.cuda.get_device_name(0) if device=="cuda" else "none")

DATA="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload"

with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)
X_traj=data["X_traj"]
y_arr=data["y_arr"]
groups=data["groups"]

print("loaded:",len(X_traj),"batches | y",y_arr.shape,"| codes",len(np.unique(groups)))
print("any NaN:",any(np.isnan(a).any() for a in X_traj))

device: cuda
gpu: Tesla T4
loaded: 1005 batches | y (1005, 4) | codes 25
any NaN: False


## Data preparation

I downsample every second step (20 second resolution) and pad or cap the trajectories at 6000 steps, which fully covers 93 percent of batches. Normalisation uses training statistics only, computed inside each fold.

In [2]:
STRIDE=2
MAXLEN=6000
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]

def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]
        a=(a-mean)/std
        if len(a)<maxlen:
            pad=np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")
            a=np.vstack([a,pad])
        else:
            a=a[:maxlen]
        out.append(a)
    arr=np.array(out,dtype="float32")
    return torch.tensor(arr).transpose(1,2)

covered=sum(1 for a in X_traj if len(a[::STRIDE])<=MAXLEN)
print("stride",STRIDE,"maxlen",MAXLEN)
print("fully covered:",covered,"of",len(X_traj),f"({100*covered/len(X_traj):.0f}%)")

stride 2 maxlen 6000
fully covered: 939 of 1005 (93%)


## Model: TCN encoder with evidential heads

The shared TCN encoder has four dilated residual blocks. Each of the four targets gets an evidential head that outputs the parameters of a Normal-Inverse-Gamma distribution, giving a prediction and its uncertainty. The evidential parameters are clamped to a safe range so the variance cannot diverge.

In [3]:
class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)
        self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout)
        self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout)
        self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x)
        return x.mean(dim=2)

class EvidentialHead(nn.Module):
    def __init__(self,in_dim,hidden=64,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,4))
    def forward(self,x):
        out=self.net(x)
        gamma=out[:,0]
        nu=F.softplus(out[:,1]).clamp(min=1e-2,max=1e3)
        alpha=F.softplus(out[:,2]).clamp(min=1e-2,max=1e3)+1.0
        beta=F.softplus(out[:,3]).clamp(min=1e-2,max=1e3)
        return gamma,nu,alpha,beta

def evidential_loss(y,gamma,nu,alpha,beta,lam=1.0):
    om=2*beta*(1+nu)
    nll=(0.5*torch.log(np.pi/nu)-alpha*torch.log(om)
         +(alpha+0.5)*torch.log(nu*(y-gamma)**2+om)
         +torch.lgamma(alpha)-torch.lgamma(alpha+0.5))
    reg=torch.abs(y-gamma)*(2*nu+alpha)
    return (nll+lam*reg).mean()

class MTTrajNet(nn.Module):
    def __init__(self,in_ch=10,hidden=128,n_targets=4,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.heads=nn.ModuleList([EvidentialHead(hidden,dropout=dropout) for _ in range(n_targets)])
    def forward(self,x):
        z=self.encoder(x)
        outs=[h(z) for h in self.heads]
        gamma=torch.stack([o[0] for o in outs],dim=1)
        nu=torch.stack([o[1] for o in outs],dim=1)
        alpha=torch.stack([o[2] for o in outs],dim=1)
        beta=torch.stack([o[3] for o in outs],dim=1)
        return gamma,nu,alpha,beta

m=MTTrajNet().to(device)
t=torch.randn(3,10,500).to(device)
g,n,a,b=m(t)
var=b/(n*(a-1))
print("predictions:",g.shape,"| variance finite:",torch.isfinite(var).all().item())
print("parameters:",sum(p.numel() for p in m.parameters()))

predictions: torch.Size([3, 4]) | variance finite: True
parameters: 384400


## One training run that produces everything

I keep the original evaluation design, grouped cross-validation on product code, which was chosen to prevent leakage before any results were seen. Within each fold the training products are split three ways: one part trains the model, one is used for early stopping, and a separate part fits the calibration scale, so the test set is never used to tune calibration. The function returns the per-fold RMSE, the test predictions and the recalibrated uncertainties, all from the same model. Seeds are fixed so the run is reproducible.

In [4]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.stats import norm
import time,warnings
warnings.filterwarnings("ignore")

def compute_ece(y,pred,std,n_bins=10):
    z=np.abs(y-pred)/std
    levels=np.linspace(0.05,0.95,n_bins)
    e=0.0
    for c in levels:
        zc=norm.ppf(0.5+c/2)
        e+=abs(np.mean(z<=zc)-c)
    return e/len(levels)

def run_final(n_splits=3,max_epochs=150,lr=5e-4,batch_size=16,patience=10):
    gkf=GroupKFold(n_splits=n_splits)
    fold_rmse=[];all_pred=[];all_true=[];all_std=[];all_code=[]
    for fold,(trv,te) in enumerate(gkf.split(X_traj,y_arr[:,0],groups=groups)):
        g=groups[trv];uniq=np.unique(g)
        rng2=np.random.RandomState(SEED+fold)
        shuf=rng2.permutation(uniq)
        k=max(1,len(shuf)//8)
        val_codes=shuf[:k];cal_codes=shuf[k:2*k]
        tr=trv[~np.isin(g,np.concatenate([val_codes,cal_codes]))]
        va=trv[np.isin(g,val_codes)]
        ca=trv[np.isin(g,cal_codes)]

        xmean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
        xstd=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
        Xtr=prep_batch([X_traj[i] for i in tr],xmean,xstd)
        Xva=prep_batch([X_traj[i] for i in va],xmean,xstd)
        Xca=prep_batch([X_traj[i] for i in ca],xmean,xstd)
        Xte=prep_batch([X_traj[i] for i in te],xmean,xstd)
        ymean=y_arr[tr].mean(axis=0);ystd=y_arr[tr].std(axis=0)+1e-6
        ytr=torch.tensor((y_arr[tr]-ymean)/ystd,dtype=torch.float32)
        yva=torch.tensor((y_arr[va]-ymean)/ystd,dtype=torch.float32)

        torch.manual_seed(SEED+fold)
        model=MTTrajNet().to(device)
        opt=torch.optim.Adam(model.parameters(),lr=lr)
        best=float("inf");best_state=None;wait=0
        for ep in range(max_epochs):
            model.train()
            perm=torch.randperm(len(Xtr))
            for i in range(0,len(Xtr),batch_size):
                idx=perm[i:i+batch_size]
                xb=Xtr[idx].to(device);yb=ytr[idx].to(device)
                opt.zero_grad()
                gg,nn_,aa,bb=model(xb)
                loss=sum(evidential_loss(yb[:,k2],gg[:,k2],nn_[:,k2],aa[:,k2],bb[:,k2]) for k2 in range(4))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step()
            model.eval()
            with torch.no_grad():
                vl=0
                for i in range(0,len(Xva),batch_size):
                    xb=Xva[i:i+batch_size].to(device);yb=yva[i:i+batch_size].to(device)
                    gg,nn_,aa,bb=model(xb)
                    vl+=sum(evidential_loss(yb[:,k2],gg[:,k2],nn_[:,k2],aa[:,k2],bb[:,k2]) for k2 in range(4)).item()
            if vl<best-1e-4:
                best=vl;best_state={k3:v.cpu().clone() for k3,v in model.state_dict().items()};wait=0
            else:
                wait+=1
                if wait>=patience:break
        model.load_state_dict(best_state);model.eval()

        def predict(X):
            gs=[];ss=[]
            with torch.no_grad():
                for i in range(0,len(X),batch_size):
                    xb=X[i:i+batch_size].to(device)
                    gg,nn_,aa,bb=model(xb)
                    v=bb/(nn_*(aa-1))
                    gs.append(gg.cpu().numpy());ss.append(torch.sqrt(v).cpu().numpy())
            return np.vstack(gs),np.vstack(ss)

        gca,sca=predict(Xca)
        pca=gca*ystd+ymean;sca_r=sca*ystd
        yca=y_arr[ca]
        scales=[np.sqrt(np.mean((pca[:,k2]-yca[:,k2])**2))/np.mean(sca_r[:,k2]) for k2 in range(4)]

        gte,ste=predict(Xte)
        pred=gte*ystd+ymean;std=ste*ystd
        for k2 in range(4):
            std[:,k2]*=scales[k2]

        yte=y_arr[te]
        rmse=[np.sqrt(mean_squared_error(yte[:,k2],pred[:,k2])) for k2 in range(4)]
        fold_rmse.append(rmse)
        all_pred.append(pred);all_true.append(yte);all_std.append(std);all_code.append(groups[te])
        print(f"fold {fold+1} RMSE:",{targets[k2]:round(rmse[k2],3) for k2 in range(4)})
    return np.array(fold_rmse),np.vstack(all_pred),np.vstack(all_true),np.vstack(all_std),np.concatenate(all_code)

print("ready")
               

ready


## The single authoritative run

I run the training once. The per-fold RMSE, the predictions and the recalibrated uncertainties all come from this one model, so the comparison table, the significance tests and the calibration metrics are consistent with each other. Because the seeds are fixed, this run is reproducible.

In [5]:
t0=time.time()
fold_rmse,pred_all,true_all,std_all,code_all=run_final(n_splits=3,max_epochs=150,batch_size=16,patience=10)
mean_rmse=fold_rmse.mean(axis=0)
std_rmse=fold_rmse.std(axis=0)
print()
print("MT-TrajNet RMSE per target (mean, fold spread):")
for k in range(4):
    print(f"  {targets[k]}: {mean_rmse[k]:.3f} (std {std_rmse[k]:.3f})  folds {[round(f,3) for f in fold_rmse[:,k]]}")
print("\ntime:",round(time.time()-t0),"seconds")

fold 1 RMSE: {'dissolution_av': np.float64(4.049), 'tbl_av_hardness': np.float64(19.713), 'tbl_rsd_weight': np.float64(0.445), 'fct_tensile': np.float64(0.376)}
fold 2 RMSE: {'dissolution_av': np.float64(3.428), 'tbl_av_hardness': np.float64(7.842), 'tbl_rsd_weight': np.float64(0.528), 'fct_tensile': np.float64(0.304)}
fold 3 RMSE: {'dissolution_av': np.float64(4.14), 'tbl_av_hardness': np.float64(11.662), 'tbl_rsd_weight': np.float64(0.688), 'fct_tensile': np.float64(0.608)}

MT-TrajNet RMSE per target (mean, fold spread):
  dissolution_av: 3.872 (std 0.316)  folds [np.float64(4.049), np.float64(3.428), np.float64(4.14)]
  tbl_av_hardness: 13.072 (std 4.948)  folds [np.float64(19.713), np.float64(7.842), np.float64(11.662)]
  tbl_rsd_weight: 0.554 (std 0.101)  folds [np.float64(0.445), np.float64(0.528), np.float64(0.688)]
  fct_tensile: 0.430 (std 0.130)  folds [np.float64(0.376), np.float64(0.304), np.float64(0.608)]

time: 188 seconds


## Calibration metrics from the same run

I now compute the coverage and the expected calibration error using the uncertainties from this same trained model, so the accuracy numbers and the calibration numbers are consistent with one another.

In [6]:
z90=norm.ppf(0.95)
print(f"{'target':<18}{'PICP(90%)':<12}{'ECE':<10}{'unc-err corr':<14}{'RMSE':<8}")
cal={}
for k in range(4):
    p=pred_all[:,k];t=true_all[:,k];s=std_all[:,k]
    picp=np.mean((t>=p-z90*s)&(t<=p+z90*s))
    ece=compute_ece(t,p,s)
    corr=np.corrcoef(s,np.abs(p-t))[0,1]
    rmse=np.sqrt(np.mean((p-t)**2))
    cal[targets[k]]={"picp":round(float(picp),3),"ece":round(float(ece),3),"corr":round(float(corr),3)}
    print(f"{targets[k]:<18}{picp:<12.3f}{ece:<10.3f}{corr:<14.3f}{rmse:<8.3f}")

target            PICP(90%)   ECE       unc-err corr  RMSE    
dissolution_av    0.863       0.034     0.200         3.885   
tbl_av_hardness   0.848       0.030     0.739         13.977  
tbl_rsd_weight    0.748       0.101     0.021         0.563   
fct_tensile       0.847       0.032     0.274         0.449   


## Save all results from this run

Everything from this single model goes into one file: the per-fold RMSE, the mean and spread, and the calibration metrics. Any table or test in the thesis uses these numbers.

In [7]:
import json

final={
"run":"MT-TrajNet final, seeded, grouped CV, held-out calibration split",
"seed":SEED,
"setup":"3-fold GroupKFold on product code, early stopping, stride-2, MAXLEN 6000, hidden 128",
"rmse_mean":{targets[k]:round(float(mean_rmse[k]),3) for k in range(4)},
"rmse_std":{targets[k]:round(float(std_rmse[k]),3) for k in range(4)},
"rmse_folds":{targets[k]:[round(float(f),3) for f in fold_rmse[:,k]] for k in range(4)},
"calibration":cal
}
with open("/kaggle/working/mttrajnet_final.json","w") as fh:
    json.dump(final,fh,indent=2)
print(json.dumps(final,indent=2))

{
  "run": "MT-TrajNet final, seeded, grouped CV, held-out calibration split",
  "seed": 42,
  "setup": "3-fold GroupKFold on product code, early stopping, stride-2, MAXLEN 6000, hidden 128",
  "rmse_mean": {
    "dissolution_av": 3.872,
    "tbl_av_hardness": 13.072,
    "tbl_rsd_weight": 0.554,
    "fct_tensile": 0.43
  },
  "rmse_std": {
    "dissolution_av": 0.316,
    "tbl_av_hardness": 4.948,
    "tbl_rsd_weight": 0.101,
    "fct_tensile": 0.13
  },
  "rmse_folds": {
    "dissolution_av": [
      4.049,
      3.428,
      4.14
    ],
    "tbl_av_hardness": [
      19.713,
      7.842,
      11.662
    ],
    "tbl_rsd_weight": [
      0.445,
      0.528,
      0.688
    ],
    "fct_tensile": [
      0.376,
      0.304,
      0.608
    ]
  },
  "calibration": {
    "dissolution_av": {
      "picp": 0.863,
      "ece": 0.034,
      "corr": 0.2
    },
    "tbl_av_hardness": {
      "picp": 0.848,
      "ece": 0.03,
      "corr": 0.739
    },
    "tbl_rsd_weight": {
      "picp": 0.74

## Train and save one model for SHAP

SHAP needs a trained model to explain. Here I train MT-TrajNet on the first fold and keep the model and that fold's test data in memory, so I can run the attribution on it. This is the gate to the RQ3b analysis.

In [8]:
gkf=GroupKFold(n_splits=3)
splits=list(gkf.split(X_traj,y_arr[:,0],groups=groups))
trv,te=splits[0]

g=groups[trv];uniq=np.unique(g)
rng2=np.random.RandomState(SEED)
shuf=rng2.permutation(uniq)
k=max(1,len(shuf)//8)
val_codes=shuf[:k]
tr=trv[~np.isin(g,val_codes)]
va=trv[np.isin(g,val_codes)]

xmean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
xstd=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
Xtr=prep_batch([X_traj[i] for i in tr],xmean,xstd)
Xva=prep_batch([X_traj[i] for i in va],xmean,xstd)
Xte=prep_batch([X_traj[i] for i in te],xmean,xstd)
ymean=y_arr[tr].mean(axis=0);ystd=y_arr[tr].std(axis=0)+1e-6
ytr=torch.tensor((y_arr[tr]-ymean)/ystd,dtype=torch.float32)
yva=torch.tensor((y_arr[va]-ymean)/ystd,dtype=torch.float32)

torch.manual_seed(SEED)
shap_model=MTTrajNet().to(device)
opt=torch.optim.Adam(shap_model.parameters(),lr=5e-4)
best=float("inf");best_state=None;wait=0
for ep in range(150):
    shap_model.train()
    perm=torch.randperm(len(Xtr))
    for i in range(0,len(Xtr),16):
        idx=perm[i:i+16]
        xb=Xtr[idx].to(device);yb=ytr[idx].to(device)
        opt.zero_grad()
        gg,nn_,aa,bb=shap_model(xb)
        loss=sum(evidential_loss(yb[:,j],gg[:,j],nn_[:,j],aa[:,j],bb[:,j]) for j in range(4))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(shap_model.parameters(),1.0)
        opt.step()
    shap_model.eval()
    with torch.no_grad():
        vl=0
        for i in range(0,len(Xva),16):
            xb=Xva[i:i+16].to(device);yb=yva[i:i+16].to(device)
            gg,nn_,aa,bb=shap_model(xb)
            vl+=sum(evidential_loss(yb[:,j],gg[:,j],nn_[:,j],aa[:,j],bb[:,j]) for j in range(4)).item()
    if vl<best-1e-4:
        best=vl;best_state={kk:v.cpu().clone() for kk,v in shap_model.state_dict().items()};wait=0
    else:
        wait+=1
        if wait>=10:break

shap_model.load_state_dict(best_state)
shap_model.eval()
torch.save(shap_model.state_dict(),"/kaggle/working/mttrajnet_fold1.pt")
print("model trained and saved, stopped at epoch",ep+1)
print("test batches for SHAP:",len(te))

model trained and saved, stopped at epoch 16
test batches for SHAP: 335


## First SHAP run: raw per-channel attributions

Here I get one working SHAP run, which is the gate to the RQ3b analysis. I use GradientExplainer, which backpropagates through the network including the pooling layer, rather than KernelSHAP, which would be far too slow on this input size. I run it on the dissolution head for a small number of test batches and report the raw attribution summed per sensor channel, before any uncertainty stratification.

In [9]:
import shap

class DissolutionWrapper(nn.Module):
    def __init__(self,model):
        super().__init__()
        self.model=model
    def forward(self,x):
        g,_,_,_=self.model(x)
        return g[:,0:1]

wrapper=DissolutionWrapper(shap_model).to(device)
wrapper.eval()

n_bg=8
n_test=8
bg=Xtr[:n_bg].to(device)
test_sample=Xte[:n_test].to(device)

explainer=shap.GradientExplainer(wrapper,bg)
shap_values=explainer.shap_values(test_sample)

sv=np.array(shap_values)
print("shap values shape:",sv.shape)

shap values shape: (8, 10, 6000, 1)


## Raw per-channel attribution values

I aggregate the attributions over time to get one value per sensor channel. This is the raw attribution before any uncertainty stratification, showing which sensors the model relies on when predicting dissolution.

In [10]:
sensors=["tbl_speed","fom","main_comp","tbl_fill","SREL",
         "pre_comp","cyl_main","cyl_pre","stiffness","ejection"]

sv=sv[:,:,:,0]
per_channel=np.abs(sv).sum(axis=2).mean(axis=0)
signed=sv.sum(axis=2).mean(axis=0)

print("raw per-channel attribution for dissolution (mean over 8 test batches)")
print(f"{'sensor':<12}{'mean |attr|':<14}{'mean signed attr'}")
for i,s in enumerate(sensors):
    print(f"{s:<12}{per_channel[i]:<14.4f}{signed[i]:.4f}")

order=np.argsort(-per_channel)
print("\nranked by magnitude:")
for r,i in enumerate(order):
    print(f"  {r+1}. {sensors[i]}: {per_channel[i]:.4f}")

raw per-channel attribution for dissolution (mean over 8 test batches)
sensor      mean |attr|   mean signed attr
tbl_speed   0.3468        0.3108
fom         0.1245        0.0904
main_comp   0.2580        -0.2352
tbl_fill    1.3310        -1.3295
SREL        0.0552        0.0205
pre_comp    1.1822        1.1822
cyl_main    0.5141        -0.5137
cyl_pre     0.1181        -0.0020
stiffness   0.0172        -0.0152
ejection    0.0234        0.0054

ranked by magnitude:
  1. tbl_fill: 1.3310
  2. pre_comp: 1.1822
  3. cyl_main: 0.5141
  4. tbl_speed: 0.3468
  5. main_comp: 0.2580
  6. fom: 0.1245
  7. cyl_pre: 0.1181
  8. SREL: 0.0552
  9. ejection: 0.0234
  10. stiffness: 0.0172


## Timing the SHAP run to plan the full analysis

Before scaling up I measure how long the attribution takes, so I know how many background and test samples the analysis can afford on this GPU.

In [11]:
import time

for n_bg_test,n_te_test in [(8,8),(16,16)]:
    bg2=Xtr[:n_bg_test].to(device)
    ts2=Xte[:n_te_test].to(device)
    ex2=shap.GradientExplainer(wrapper,bg2)
    t0=time.time()
    _=ex2.shap_values(ts2)
    dt=time.time()-t0
    print(f"bg={n_bg_test}, test={n_te_test}: {dt:.1f}s  ({dt/n_te_test:.2f}s per test sample)")

bg=8, test=8: 7.7s  (0.96s per test sample)
bg=16, test=16: 15.3s  (0.96s per test sample)


## Full SHAP on the test fold, with the model's uncertainty

I run the attribution on the whole test fold rather than a subsample. I also collect the model's epistemic uncertainty for each test batch, so I can split the predictions into high and low confidence groups and compare their attribution patterns.

In [12]:
bg=Xtr[:32].to(device)
explainer=shap.GradientExplainer(wrapper,bg)

t0=time.time()
sv_all=[]
for i in range(0,len(Xte),16):
    chunk=Xte[i:i+16].to(device)
    sv_chunk=explainer.shap_values(chunk)
    sv_all.append(np.array(sv_chunk)[:,:,:,0])
sv_full=np.concatenate(sv_all,axis=0)
print("full shap values:",sv_full.shape,"time:",round(time.time()-t0),"s")

unc=[]
with torch.no_grad():
    for i in range(0,len(Xte),16):
        xb=Xte[i:i+16].to(device)
        g,n,a,b=shap_model(xb)
        epi=(1.0/n[:,0]).cpu().numpy()
        unc.append(epi)
unc=np.concatenate(unc)
print("uncertainty collected:",unc.shape)

full shap values: (335, 10, 6000) time: 321 s
uncertainty collected: (335,)


## Uncertainty-stratified attribution (RQ3b)

I split the test predictions into the most confident twenty percent and the least confident twenty percent, using the model's epistemic uncertainty. For each group I average the attributions per sensor channel and per compression phase (early, mid and late thirds of the trajectory). I then compare the two attribution maps with Kendall rank correlation, to test whether the model relies on different sensors and phases when it is uncertain.

In [13]:
from scipy.stats import kendalltau

n=len(unc)
k=int(0.2*n)
order=np.argsort(unc)
low_idx=order[:k]
high_idx=order[-k:]

print("low uncertainty group:",len(low_idx),"batches | high uncertainty group:",len(high_idx))
print("mean epistemic: low",round(float(unc[low_idx].mean()),4),"| high",round(float(unc[high_idx].mean()),4))
print()

third=sv_full.shape[2]//3
phases={"early":slice(0,third),"mid":slice(third,2*third),"late":slice(2*third,sv_full.shape[2])}

def attr_map(idx):
    m=np.zeros((10,3))
    for pi,(pname,sl) in enumerate(phases.items()):
        m[:,pi]=np.abs(sv_full[idx][:,:,sl]).sum(axis=2).mean(axis=0)
    return m

low_map=attr_map(low_idx)
high_map=attr_map(high_idx)

print("per-channel attribution, confident vs uncertain (summed over phases)")
print(f"{'sensor':<12}{'confident':<14}{'uncertain':<14}{'ratio'}")
lc=low_map.sum(axis=1); hc=high_map.sum(axis=1)
for i,s in enumerate(sensors):
    r=hc[i]/lc[i] if lc[i]>0 else float("nan")
    print(f"{s:<12}{lc[i]:<14.4f}{hc[i]:<14.4f}{r:.2f}")

tau,p=kendalltau(lc,hc)
print(f"\nKendall tau between confident and uncertain channel rankings: {tau:.3f} (p={p:.4f})")

low uncertainty group: 67 batches | high uncertainty group: 67
mean epistemic: low 26.7176 | high 90.3593

per-channel attribution, confident vs uncertain (summed over phases)
sensor      confident     uncertain     ratio
tbl_speed   0.1385        0.2748        1.98
fom         0.0776        0.1319        1.70
main_comp   0.5277        0.2514        0.48
tbl_fill    0.7539        1.4245        1.89
SREL        0.0303        0.0513        1.69
pre_comp    0.2194        0.6787        3.09
cyl_main    0.2014        0.5616        2.79
cyl_pre     0.3192        0.1045        0.33
stiffness   0.0221        0.0208        0.94
ejection    0.0438        0.0280        0.64

Kendall tau between confident and uncertain channel rankings: 0.644 (p=0.0091)


## Attribution by compression phase

I now split the attribution by phase of the compression trajectory, early, mid and late thirds, to see whether confident and uncertain predictions rely on different stages of the process.

In [14]:
print("attribution by phase (summed over channels)")
print(f"{'phase':<10}{'confident':<14}{'uncertain':<14}{'ratio'}")
lp=low_map.sum(axis=0); hp=high_map.sum(axis=0)
for i,pname in enumerate(phases.keys()):
    r=hp[i]/lp[i] if lp[i]>0 else float("nan")
    print(f"{pname:<10}{lp[i]:<14.4f}{hp[i]:<14.4f}{r:.2f}")

tau_p,p_p=kendalltau(lp,hp)
print(f"\nKendall tau between confident and uncertain phase rankings: {tau_p:.3f} (p={p_p:.4f})")

print("\ntop channel-phase combinations")
for label,m in [("confident",low_map),("uncertain",high_map)]:
    flat=[(sensors[i],list(phases.keys())[j],m[i,j]) for i in range(10) for j in range(3)]
    flat.sort(key=lambda x:-x[2])
    print(f"  {label}:")
    for s,ph,v in flat[:4]:
        print(f"    {s} / {ph}: {v:.4f}")

attribution by phase (summed over channels)
phase     confident     uncertain     ratio
early     2.1597        2.2139        1.03
mid       0.0946        0.9556        10.10
late      0.0796        0.3579        4.49

Kendall tau between confident and uncertain phase rankings: 1.000 (p=0.3333)

top channel-phase combinations
  confident:
    tbl_fill / early: 0.6745
    main_comp / early: 0.5197
    cyl_pre / early: 0.3024
    pre_comp / early: 0.2147
  uncertain:
    tbl_fill / early: 0.8817
    pre_comp / early: 0.5037
    tbl_fill / mid: 0.3952
    cyl_main / early: 0.3297


## Save the RQ3b attribution results

I save the attribution maps and the comparison statistics so the analysis is recorded. Note that the phase-level Kendall test has almost no power with only three categories, so I report the phase ratios rather than that statistic.

In [15]:
rq3b={
"analysis":"RQ3b uncertainty-stratified SHAP attribution, fold 1, dissolution target",
"setup":"GradientExplainer, 32 background samples, full test fold (335 batches), top and bottom 20 percent by epistemic uncertainty",
"group_sizes":{"confident":int(len(low_idx)),"uncertain":int(len(high_idx))},
"mean_epistemic":{"confident":round(float(unc[low_idx].mean()),3),"uncertain":round(float(unc[high_idx].mean()),3)},
"per_channel":{sensors[i]:{"confident":round(float(lc[i]),4),"uncertain":round(float(hc[i]),4),"ratio":round(float(hc[i]/lc[i]),3)} for i in range(10)},
"per_phase":{list(phases.keys())[i]:{"confident":round(float(lp[i]),4),"uncertain":round(float(hp[i]),4),"ratio":round(float(hp[i]/lp[i]),3)} for i in range(3)},
"kendall_tau_channels":{"tau":round(float(tau),3),"p":round(float(p),4)},
"note":"channel rankings differ significantly between confident and uncertain predictions; phase attention shifts from almost entirely early phase when confident to also mid and late phases when uncertain; phase-level Kendall test has no power with three categories"
}
import json
with open("/kaggle/working/rq3b_attribution.json","w") as fh:
    json.dump(rq3b,fh,indent=2)
print(json.dumps(rq3b,indent=2))

{
  "analysis": "RQ3b uncertainty-stratified SHAP attribution, fold 1, dissolution target",
  "setup": "GradientExplainer, 32 background samples, full test fold (335 batches), top and bottom 20 percent by epistemic uncertainty",
  "group_sizes": {
    "confident": 67,
    "uncertain": 67
  },
  "mean_epistemic": {
    "confident": 26.718,
    "uncertain": 90.359
  },
  "per_channel": {
    "tbl_speed": {
      "confident": 0.1385,
      "uncertain": 0.2748,
      "ratio": 1.984
    },
    "fom": {
      "confident": 0.0776,
      "uncertain": 0.1319,
      "ratio": 1.701
    },
    "main_comp": {
      "confident": 0.5277,
      "uncertain": 0.2514,
      "ratio": 0.476
    },
    "tbl_fill": {
      "confident": 0.7539,
      "uncertain": 1.4245,
      "ratio": 1.89
    },
    "SREL": {
      "confident": 0.0303,
      "uncertain": 0.0513,
      "ratio": 1.691
    },
    "pre_comp": {
      "confident": 0.2194,
      "uncertain": 0.6787,
      "ratio": 3.094
    },
    "cyl_main": {
 

## Summary of this notebook

Everything below comes from the single seeded run in this notebook: the accuracy, the calibration, and the attribution analysis. Keeping them from one model means the comparison table and the statistical tests are consistent with each other.

In [16]:
print("MT-TRAJNET FINAL RUN (seed 42)\n")
print("Accuracy, RMSE per target (mean, fold spread):")
for k in range(4):
    print(f"  {targets[k]:<18}{mean_rmse[k]:.3f} (std {std_rmse[k]:.3f})  folds {[round(f,3) for f in fold_rmse[:,k]]}")

print("\nCalibration:")
print(f"  {'target':<18}{'PICP(90%)':<12}{'ECE':<10}{'unc-err corr'}")
for k in range(4):
    c=cal[targets[k]]
    print(f"  {targets[k]:<18}{c['picp']:<12}{c['ece']:<10}{c['corr']}")

print("\nRQ3b attribution, dissolution, fold 1:")
print(f"  Kendall tau between confident and uncertain channel rankings: {tau:.3f} (p={p:.4f})")
print("  phase attention ratio, uncertain over confident:")
for i,pname in enumerate(phases.keys()):
    print(f"    {pname}: {hp[i]/lp[i]:.2f}x")
print("\n  channels relied on more when uncertain:")
for i in np.argsort(-(hc/lc))[:3]:
    print(f"    {sensors[i]}: {hc[i]/lc[i]:.2f}x")
print("  channels relied on less when uncertain:")
for i in np.argsort(hc/lc)[:3]:
    print(f"    {sensors[i]}: {hc[i]/lc[i]:.2f}x")

MT-TRAJNET FINAL RUN (seed 42)

Accuracy, RMSE per target (mean, fold spread):
  dissolution_av    3.872 (std 0.316)  folds [np.float64(4.049), np.float64(3.428), np.float64(4.14)]
  tbl_av_hardness   13.072 (std 4.948)  folds [np.float64(19.713), np.float64(7.842), np.float64(11.662)]
  tbl_rsd_weight    0.554 (std 0.101)  folds [np.float64(0.445), np.float64(0.528), np.float64(0.688)]
  fct_tensile       0.430 (std 0.130)  folds [np.float64(0.376), np.float64(0.304), np.float64(0.608)]

Calibration:
  target            PICP(90%)   ECE       unc-err corr
  dissolution_av    0.863       0.034     0.2
  tbl_av_hardness   0.848       0.03      0.739
  tbl_rsd_weight    0.748       0.101     0.021
  fct_tensile       0.847       0.032     0.274

RQ3b attribution, dissolution, fold 1:
  Kendall tau between confident and uncertain channel rankings: 0.644 (p=0.0091)
  phase attention ratio, uncertain over confident:
    early: 1.03x
    mid: 10.10x
    late: 4.49x

  channels relied on more 